In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import natsort
from sklearn.preprocessing import StandardScaler

# 경로 설정
train_path = '/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/train/'
test_path = '/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/test/'

train_list = natsort.natsorted(os.listdir(train_path))
test_list = natsort.natsorted(os.listdir(test_path))

In [2]:
# train.csv 파일 읽기
train_metadata = pd.read_csv("/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/train.csv")
train_metadata = train_metadata.sort_values(['id', 'path', 'label']).reset_index(drop=True)
train_metadata

,id,path,label
0,AAACWKPZ,./train/AAACWKPZ.ogg,fake
1,AAAQOZYI,./train/AAAQOZYI.ogg,fake
2,AAAWBXQE,./train/AAAWBXQE.ogg,real
3,AABHDRLX,./train/AABHDRLX.ogg,real
4,AABXXHMU,./train/AABXXHMU.ogg,real
...,...,...,...
55433,ZZZMJSBX,./train/ZZZMJSBX.ogg,fake
55434,ZZZRTSKN,./train/ZZZRTSKN.ogg,fake
55435,ZZZSAANM,./train/ZZZSAANM.ogg,real
55436,ZZZSYEYS,./train/ZZZSYEYS.ogg,fake


In [3]:
train_list

['AAACWKPZ.ogg',
 'AAAQOZYI.ogg',
 'AAAWBXQE.ogg',
 'AABHDRLX.ogg',
 'AABXXHMU.ogg',
 'AACAYDKC.ogg',
 'AACBAWMZ.ogg',
 'AACEAXVU.ogg',
 'AACHVERO.ogg',
 'AACYAWEH.ogg',
 'AADDNUMU.ogg',
 'AADKWHEE.ogg',
 'AADPDAER.ogg',
 'AADQUEFI.ogg',
 'AAECEROJ.ogg',
 'AAEYTHSY.ogg',
 'AAFFVAWD.ogg',
 'AAFNYPXE.ogg',
 'AAGDMPXZ.ogg',
 'AAGIITHH.ogg',
 'AAGXILIN.ogg',
 'AAGZDQDL.ogg',
 'AAHKQWBB.ogg',
 'AAIGXLUD.ogg',
 'AAIINGWH.ogg',
 'AAIKZLBQ.ogg',
 'AAIXOVVV.ogg',
 'AAIYWHUF.ogg',
 'AAJEQLVT.ogg',
 'AAJKXPIW.ogg',
 'AAJMPRLS.ogg',
 'AAJSFGQB.ogg',
 'AAKKEAOL.ogg',
 'AAKXVWXI.ogg',
 'AALITDLH.ogg',
 'AALMMBOZ.ogg',
 'AALVLTSV.ogg',
 'AAMMREFE.ogg',
 'AANJJHYA.ogg',
 'AANOXTME.ogg',
 'AANUFJBH.ogg',
 'AAOEAZJW.ogg',
 'AAOFKKOG.ogg',
 'AAOYUWCN.ogg',
 'AAPELQFY.ogg',
 'AAPGWSKV.ogg',
 'AAPWJSUB.ogg',
 'AAQBLCRK.ogg',
 'AAQBNHMD.ogg',
 'AAQBQXCT.ogg',
 'AAQFLJFN.ogg',
 'AAQJRBNN.ogg',
 'AAQNTEKV.ogg',
 'AAQQXFSY.ogg',
 'AARBSIOH.ogg',
 'AARGVJZI.ogg',
 'AARGWRKZ.ogg',
 'AARRJAAQ.ogg',
 'AASWUMSW.ogg

In [4]:
Y=pd.DataFrame(train_metadata['label'],columns=['label'])
Y

,label
0,fake
1,fake
2,real
3,real
4,real
...,...
55433,fake
55434,fake
55435,real
55436,fake


In [5]:
# 데이터 증강 함수
import random
def augment_audio(y):
    # 시간 축소
    if random.random() < 0.5:
        y = librosa.effects.time_stretch(y, rate=random.uniform(0.8, 1.2))
    # 피치 변경
    if random.random() < 0.5:
        y = librosa.effects.pitch_shift(y, sr=sample_rate, n_steps=random.randint(-3, 3))
    return y

In [6]:
# Constants
duration = 5  # seconds
sample_rate = 32000  # Hz
n_mels = 128
n_mfcc = 40


In [7]:
# Initialize lists for features and labels
train_feature = []

for i in range(len(train_list)):
    x, _ = librosa.load(train_path + train_list[i], sr=sample_rate, duration=duration)
    
    # 데이터 증강
    x = augment_audio(x)
    
    # 노이즈 감소
    x = librosa.effects.preemphasis(x)
    
    # Extract log-mel spectrogram
    S = librosa.feature.melspectrogram(y=x, sr=sample_rate, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Extract MFCC
    mfcc = librosa.feature.mfcc(S=log_S, n_mfcc=n_mfcc)
    
    # Compute delta and delta-delta MFCC
    delta_mfcc = librosa.feature.delta(mfcc, width=min(7, mfcc.shape[1]))
    delta2_mfcc = librosa.feature.delta(mfcc, order=2, width=min(7, mfcc.shape[1]))
    
    # Extract spectral contrast
    spectral_contrast = librosa.feature.spectral_contrast(S=log_S, sr=sample_rate)
    
    # Concatenate features along the MFCC axis
    combined_features = np.concatenate((mfcc, delta_mfcc, delta2_mfcc, spectral_contrast), axis=0)
    
    # Normalize the combined features
    scaler = StandardScaler()
    normalized_features = scaler.fit_transform(combined_features)
    
    # Append to the feature list
    train_feature.append(normalized_features.flatten())


: 

In [8]:
train_feature_df = pd.DataFrame(train_feature)
train_feature_df

In [ ]:
train_feature_df=train_feature_df.dropna(axis=1)

In [ ]:
train_feature_df

,0,1,2,3,4,5,6,7,8,9,...,830,831,832,833,834,835,836,837,838,839
0,-8.559496,-7.928725,-7.672396,-7.676212,-8.218899,-9.075097,-9.405861,-9.217286,-8.694652,-8.052616,...,-0.143670,-0.166276,-0.175169,-0.168969,-0.154302,-0.138895,-0.175562,-0.119635,0.109827,0.167286
1,-9.920085,-10.769081,-10.684294,-9.328842,-8.532884,-8.197854,-8.346693,-8.634234,-9.192903,-10.348639,...,0.103175,0.200001,0.496926,0.629411,0.500303,0.345103,0.306861,0.283479,0.167817,0.196016
2,-9.913172,-9.387213,-8.902587,-8.340944,-7.921254,-7.893692,-8.247587,-9.108587,-10.301409,-9.692926,...,0.120113,0.010279,0.089210,0.177160,0.177490,0.255475,0.103873,0.170995,0.182634,0.098724
3,-8.531118,-7.809933,-8.061981,-8.875454,-9.863110,-10.466659,-10.302869,-10.310455,-10.597152,-10.346334,...,0.963978,1.512708,1.505057,1.111069,0.869279,0.729185,0.641078,0.479249,0.370862,0.120363
4,-8.868017,-8.676674,-9.344377,-9.751315,-9.840075,-9.942251,-9.782149,-9.064535,-8.421437,-8.250699,...,-0.179680,0.186828,0.339651,0.368933,0.452388,0.448328,0.336424,0.219360,0.183254,0.327383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55433,-10.375456,-11.470290,-12.173081,-9.812245,-9.192671,-9.358091,-9.446752,-9.431613,-9.317065,-9.434489,...,0.054571,0.229986,0.254080,0.188672,-0.001761,-0.137309,-0.084373,-0.047884,0.161755,0.043360
55434,-9.597319,-9.108658,-8.842084,-8.667772,-8.879213,-9.170095,-9.639820,-10.120221,-10.175445,-9.919191,...,0.431816,0.496441,0.430751,0.378575,0.394933,0.496851,0.552124,0.578591,0.559384,0.543076
55435,-5.017020,-5.454221,-7.467025,-8.111515,-8.584310,-9.537187,-11.268817,-12.754025,-13.523101,-13.573894,...,0.247223,0.270697,0.356332,0.572925,0.459568,0.086830,-0.069607,-0.150953,-0.180580,-0.188704
55436,-11.811006,-11.269548,-10.822845,-10.494061,-10.413902,-10.533603,-10.702593,-11.061482,-11.306023,-10.244934,...,-0.037024,0.158707,0.134981,0.077331,0.107578,0.291742,0.399695,0.165196,0.258710,0.287106


In [ ]:
save_data=pd.concat([train_feature_df,Y],axis=1)

In [ ]:
save_data.to_csv('train data0708(40).csv')

In [ ]:
train_feature_df2 = pd.DataFrame(train_feature)

In [ ]:
train_feature_df2

,0,1,2,3,4,5,6,7,8,9,...,37550,37551,37552,37553,37554,37555,37556,37557,37558,37559
0,-8.559496,-7.928725,-7.672396,-7.676212,-8.218899,-9.075097,-9.405861,-9.217286,-8.694652,-8.052616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-9.920085,-10.769081,-10.684294,-9.328842,-8.532884,-8.197854,-8.346693,-8.634234,-9.192903,-10.348639,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-9.913172,-9.387213,-8.902587,-8.340944,-7.921254,-7.893692,-8.247587,-9.108587,-10.301409,-9.692926,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-8.531118,-7.809933,-8.061981,-8.875454,-9.863110,-10.466659,-10.302869,-10.310455,-10.597152,-10.346334,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-8.868017,-8.676674,-9.344377,-9.751315,-9.840075,-9.942251,-9.782149,-9.064535,-8.421437,-8.250699,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55433,-10.375456,-11.470290,-12.173081,-9.812245,-9.192671,-9.358091,-9.446752,-9.431613,-9.317065,-9.434489,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55434,-9.597319,-9.108658,-8.842084,-8.667772,-8.879213,-9.170095,-9.639820,-10.120221,-10.175445,-9.919191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55435,-5.017020,-5.454221,-7.467025,-8.111515,-8.584310,-9.537187,-11.268817,-12.754025,-13.523101,-13.573894,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55436,-11.811006,-11.269548,-10.822845,-10.494061,-10.413902,-10.533603,-10.702593,-11.061482,-11.306023,-10.244934,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
train_feature_df2.to_csv("train data isnull0708(40).csv")

In [ ]:
train_feature_df3=train_feature_df2.dropna(axis=0)
train_feature_df3

,0,1,2,3,4,5,6,7,8,9,...,37550,37551,37552,37553,37554,37555,37556,37557,37558,37559
6,-7.469420,-7.713595,-8.838909,-9.204244,-9.984107,-10.771160,-10.906266,-10.928009,-10.986822,-11.052005,...,0.067899,0.060177,0.042838,0.051408,0.066441,0.073136,0.065752,0.065752,0.065752,0.065752
7,-8.619802,-8.605498,-8.890416,-9.011770,-9.203639,-9.332529,-9.444480,-9.872317,-10.451763,-10.725849,...,0.102369,0.110299,0.120276,0.104176,0.085866,0.088102,0.088947,0.088947,0.088947,0.088947
17,-8.711096,-8.953998,-9.683305,-9.749688,-9.788881,-9.809483,-9.850435,-9.496692,-9.027873,-8.769351,...,0.114462,0.093346,0.049954,0.065325,0.133205,0.186275,0.162992,0.162992,0.162992,0.162992
29,-7.574636,-7.490919,-7.928205,-8.104813,-8.328063,-8.525185,-8.791017,-9.235061,-9.889820,-10.288619,...,0.123877,0.124659,0.130740,0.136535,0.136561,0.105988,0.039125,0.039125,0.039125,0.039125
35,-9.503181,-9.242467,-8.980169,-9.231420,-9.564940,-9.774562,-9.978538,-10.302010,-10.575899,-11.064992,...,0.104743,0.120601,0.110323,0.098849,0.078819,0.089783,0.103930,0.103930,0.103930,0.103930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55411,-9.519038,-8.961894,-9.143313,-9.477632,-9.723955,-9.980649,-10.236880,-10.323758,-10.168305,-10.036110,...,0.082199,0.060692,0.049546,0.078934,0.108186,0.125109,0.114169,0.114169,0.114169,0.114169
55415,-9.344987,-9.853269,-10.731838,-10.582877,-10.211473,-10.003655,-9.975724,-9.965261,-9.683512,-9.396098,...,0.087868,0.089651,0.103274,0.099778,0.094847,0.092679,0.112137,0.112137,0.112137,0.112137
55418,-8.865733,-8.289367,-8.211742,-8.240948,-8.293381,-8.450284,-8.551821,-8.586887,-8.822410,-9.245096,...,0.067494,0.128493,0.118864,0.099265,0.086669,0.097613,0.109946,0.109946,0.109946,0.109946
55419,-10.399850,-10.214069,-10.212853,-9.862664,-9.487594,-9.096235,-9.182699,-9.384059,-9.438869,-9.698727,...,0.104453,0.103985,0.105815,0.109309,0.100997,0.103375,0.102353,0.102353,0.102353,0.102353


In [ ]:
train_feature_df3.to_csv("train data null(row)0708(40).csv")